# 04 Explaining A2 (WavLM + layer-weighted head)
A2 averages each of WavLM's 13 layers over time, mixes them with learned weights, then a small MLP. Time is averaged away, so instead of frames we look at layers and at silence.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from audio_df.data.manifests import load_manifest
from audio_df.helpers.eval_audio import trimmed_manifest
from audio_df.helpers.metrics import compute_eer, min_dcf
from audio_df.helpers.ssl_cache import cached_pooled, iter_pooled_chunks
from audio_df.modules.ssl_detector import SSLDetector

CACHE = Path("../data/cache").resolve()  # absolute, so cache identities match the runtime's
art = Path("../artifacts/a2/20261023-200629-150334")  # trimmed-trained A2, tracked in git
prep = art / "preprocessing.json"
train_audio = json.loads(prep.read_text())["train"] if prep.exists() else "raw"
a2 = SSLDetector.load(art, device="auto")  # GPU (if any) only for building missing WavLM caches
a2.head.cpu()
print(art.name, "| trained on", train_audio, "audio")

def eer(scores, labels):
    return round(100 * compute_eer(scores, labels, higher_is_spoof=False)[0], 2)

def feats(df, split):  # cached [clips, 13 layers, 768], no WavLM run when cached
    return cached_pooled(df, a2.encoder, CACHE / "wavlm-base-plus/pooled", split).astype(np.float32)

train, dev = load_manifest("../data/manifests/train.csv", ".."), load_manifest("../data/manifests/dev.csv", "..")
if train_audio == "trimmed":
    train = trimmed_manifest(train, CACHE, "train")
sets = {"train": train, "dev raw": dev, "dev trimmed": trimmed_manifest(dev, CACHE, "dev")}
X = {k: feats(v, "train" if k == "train" else "dev") for k, v in sets.items()}
y = {k: v.label.to_numpy() for k, v in sets.items()}

## 1 Learned layer weights

In [ ]:
w = a2.head.layer_weights().detach().numpy()
plt.figure(figsize=(7, 2.5))
plt.bar(range(13), w)
plt.axhline(1 / 13, color="grey", ls=":", label="uniform")
plt.xlabel("WavLM layer (0 = CNN output)")
plt.ylabel("weight")
plt.legend()
plt.show()

Almost uniform (0.064–0.092), slightly more on layers 0–6 than 7–12. The head barely picks a layer, it mostly averages all 13.

## 2 What does each layer know?
A linear probe per layer, trained on train, scored on raw and trimmed dev. Dashed lines = A2 itself.

In [ ]:
def fit_probe(layer):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1, class_weight="balanced"))
    return clf.fit(X["train"][:, layer], y["train"] == "bonafide")  # positive class = bona fide, like A2

def a2_scores(x):
    with torch.inference_mode():
        return a2.head(torch.from_numpy(x)).numpy()

probes = [fit_probe(l) for l in range(13)]
layers = pd.DataFrame({l: {k: eer(probes[l].decision_function(X[k][:, l]), y[k]) for k in ["dev raw", "dev trimmed"]}
                       for l in range(13)}).T
full = {k: eer(a2_scores(X[k]), y[k]) for k in ["dev raw", "dev trimmed"]}
ax = layers.plot(marker="o", figsize=(7, 3), ylabel="EER %", xlabel="WavLM layer")
for k, c in zip(full, ["C0", "C1"]):
    ax.axhline(full[k], color=c, ls="--")
plt.show()
layers.T.assign(A2=pd.Series(full))

On trimmed dev the middle layers are best: layer 4 gets 1.8%, layers 3–6 stay under 2.5%, close to A2 itself (1.4%). Both ends are weaker: layer 0 (CNN output) 6.8%, layers 9–12 about 6%. Layer 12 is what round 2's D2/D3 used.

## 2b Which layer holds up on every test set?
The same 13 probes (trained once, on train) scored on 2019 eval, its degraded copies (MP3, MUSAN noise, Opus, 8 kHz), In-the-Wild and ASVspoof 2021 LA. Missing WavLM caches are built first. minDCF uses the project costs (miss 1, false alarm 10, spoof prior 0.05).

In [ ]:
TESTS = ["eval", "eval_mp3_32k", "eval_noise_10db", "eval_opus_16k", "eval_resample_8k", "itw",
         "asvspoof2021_eval", "asvspoof2021_hidden"]

def test_scores(name):  # stream cache chunks: 13 probe scores + A2's score per clip
    df = load_manifest(f"../data/manifests/{name}.csv", "..")
    if name == "eval" or name.startswith("eval_"):  # 2019 sets are trimmed, like the runtime
        df = trimmed_manifest(df, CACHE, name)
    parts = [[] for _ in range(14)]
    for chunk in iter_pooled_chunks(df, a2.encoder, CACHE / "wavlm-base-plus/pooled", name):
        x = np.asarray(chunk, np.float32)
        for l in range(13):
            parts[l].append(probes[l].decision_function(x[:, l]))
        parts[13].append(a2_scores(x))
    return df.label.to_numpy(), [np.concatenate(p) for p in parts]

cols = [*range(13), "A2"]
EER, DCF = pd.DataFrame(columns=cols, dtype=float), pd.DataFrame(columns=cols, dtype=float)
for name in TESTS:
    labels, scores = test_scores(name)
    EER.loc[name] = [eer(s, labels) for s in scores]
    DCF.loc[name] = [round(min_dcf(s, labels, higher_is_spoof=False)[0], 3) for s in scores]
for t in (EER, DCF):
    t.loc["mean"] = t.mean()

COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharex=True)
for ax, t, label in [(axes[0], EER, "EER %"), (axes[1], DCF, "minDCF")]:
    for name, color in zip(TESTS, COLORS):
        ax.plot(range(13), t.loc[name, list(range(13))], marker="o", ms=4, lw=1.5, color=color, label=name)
    ax.plot(range(13), t.loc["mean", list(range(13))], color="#0b0b0b", lw=2.5, label="mean")
    ax.set_xticks(range(13))
    ax.set_xlabel("WavLM layer")
    ax.set_ylabel(label)
    ax.grid(axis="y", color="#e4e3df")
axes[1].legend(fontsize=8, frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.tight_layout()
plt.show()
print("best layer by mean EER:", EER.loc["mean", list(range(13))].idxmin(),
      "| by mean minDCF:", DCF.loc["mean", list(range(13))].idxmin())
EER.round(2)

In [ ]:
DCF

Dev alone picks layer 4, but averaged over all 8 test sets layer 6 is best by both metrics (mean EER 13.0%, minDCF 0.335), about as good as A2's full mix (12.9%, 0.311) and better than it on 2021 LA eval (12.4% vs 14.3%). Layer 5 is a close second and the best single layer under MUSAN noise (16.3%). Layer 12 averages 17.2%. Round 3 D2/D3 use layer 6.

## 3 Silence test
Same A2, same dev clips, raw vs trimmed audio.

In [ ]:
s = {k: a2_scores(X[k]) for k in ["dev raw", "dev trimmed"]}
shift = s["dev trimmed"] - s["dev raw"]
print(pd.DataFrame({k: pd.Series(v).groupby(y[k]).mean() for k, v in s.items()}).round(2))
plt.figure(figsize=(7, 2.5))
for lab, color in [("bonafide", "tab:green"), ("spoof", "tab:orange")]:
    plt.hist(shift[y["dev raw"] == lab], bins=80, density=True, histtype="step", color=color, label=lab)
plt.axvline(0, color="grey", lw=0.5)
plt.xlabel("score change after trimming (- = more spoof)")
plt.legend()
plt.show()

Trimming barely moves either class now: bona fide 9.40 → 9.44, spoof -14.8 → -21.0 (more spoof-like). Training on trimmed audio removed the silence shortcut; the round 1 A2 dropped about 11 points on bona fide here.

## Conclusion
- The middle WavLM layers carry the most spoof information; the last layer (12, used by round 2's
  D2/D3) is among the weakest. Over all 8 test sets layer 6 is best, so round 3 uses it.
- No layer gets In-the-Wild below 20.7% EER, and mixing all 13 (A2) only reaches 19.2%. Every layer is
  weak there in the same way, so a wider layer mix (e.g. 3–6) would not fix it. The limit is what frozen
  WavLM learns from 2019 LA; augmentation, fine-tuning WavLM or more varied training data would help.